## **Preprocessing of Train Data**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### **Clinical Data**

In [ ]:
clinical_train=pd.read_csv("/content/drive/My Drive/x_train/features/clinical_data_train.csv", index_col='PatientID')
clinical_train.head(5)

In [ ]:
clinical_train.describe()

Variable "age" has 26 missing values.

In [ ]:
clinical_train.Mstage.value_counts(sort=False, dropna=False)

Only 6 subjects have a different value to "0" for Mstage. This variable will not be used in the Cox models because there is almost no variation (=constant) and may cause convergence issue.

In [ ]:
clinical_train.Nstage.value_counts(sort=False, dropna=False)

In [ ]:
clinical_train.Tstage.value_counts(sort=False, dropna=False)

There is only one patient with Tstage="5". 
This patient is grouped with patients Tstage="4"

In [ ]:
clinical_train['Tstage']=clinical_train.Tstage.apply(lambda x: 4 if x==5 else x)

In [ ]:
clinical_train.Histology.value_counts(sort=False, dropna=False)

Similar histologies but named differently are grouped together in a new variable.

In [ ]:
clinical_train['Histology_cat']=clinical_train.Histology.apply(lambda x: 0 if x in ('Adenocarcinoma','adenocarcinoma') 
                                                               else 1 if x=='large cell' 
                                                               else 2 if x in('squamous cell carcinoma', 'Squamous cell carcinoma') 
                                                               else 3)

#Missing values and not specified histologies (NOS) are grouped together.

In [ ]:
clinical_train.Histology_cat.value_counts(sort=False, dropna=False)

Dummification of categorical variables: Tstage, Nstage and Histology_cat

In [ ]:
#Mstage and Histology variables which will not be used in the survival analysis are removed.
clinical_train=clinical_train.drop(columns=["Mstage", "Histology"])

categories = ['Nstage', 'Tstage', 'Histology_cat']
clinical_train = pd.get_dummies(clinical_train, columns=categories, drop_first=True)
clinical_train.head()

### **Radiomics Features**

In [ ]:
radiomics_train=pd.read_csv("/content/drive/My Drive/x_train/features/radiomics_train.csv", header=1, index_col='PatientID')
radiomics_train.describe()

All the 53 radiomic features are continuous variables. They are all complete (no missing data).

**Merge of clinical data and radiomics features**

In [ ]:
x_train_svm=pd.merge(radiomics_train, clinical_train, how='inner', left_index=True, right_index=True, sort=True)

Transformation Pipelines

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Dummy variables are not going to be transformed and are thus removed temporarily from the dataset.
x_train=x_train_svm.drop(columns=['Nstage_1','Nstage_2','Nstage_3','Tstage_2','Tstage_3','Tstage_4', 'Histology_cat_1', 'Histology_cat_2','Histology_cat_3'])

# Create pipeline for categorical features
categorical_features_1 = ['SourceDataset']

categorical_transformer_1 = Pipeline(
    steps=[('encoder', OneHotEncoder(drop='first'))])
#Variable "SourceDataset" is converted into one-hot vector.

# Create pipeline for numeric features
numeric_features_1 = x_train.select_dtypes(include=['int64','float64']).columns

numeric_transformer_1 = Pipeline(
    steps=[('imputer', SimpleImputer(strategy="median")),('scaler', StandardScaler())
])
#Missing values of numeric variables are replaced by its median (only "age" has missing values)
#Numeric variables are standardized (mean=0, standard deviation=1)

# Use ColumnTranformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num_1', numeric_transformer_1, numeric_features_1),
        ('cat', categorical_transformer_1, categorical_features_1)
    ])

x_train_ = preprocessor.fit_transform(x_train)
x_train_

In [ ]:
x_train=pd.DataFrame(x_train_, index=x_train.index, columns=x_train.columns)
#Dummy variables are included in a new dataset in order to be merged with the other variables afterward
test=clinical_train.loc[:,['Nstage_1','Nstage_2','Nstage_3','Tstage_2','Tstage_3','Tstage_4','Histology_cat_1', 'Histology_cat_2','Histology_cat_3']]

#Merge of the transformed variables with the dummy variables
x_train=pd.merge(x_train, test, how='inner', left_index=True, right_index=True, sort=True)
x_train=x_train.rename(columns={'SourceDataset':'age','age':'SourceDataset'})
x_train.head()

## **Target and Censorship Variables**

In [ ]:
y_train = pd.read_csv('/content/drive/My Drive/y_train.csv', index_col=0)
y_train=y_train.sort_index()
y_train.head()

**Censorship Variable**

In [ ]:
y_train.Event.value_counts(sort=False, dropna=False)

For 162 patients, the death has been observed, while 138 "escaped" from the study.

**Target Variable (Survival Time)**

In [ ]:
y_train['SurvivalTime'].describe()

In [ ]:
graph = sns.displot(y_train.SurvivalTime, bins=7)
graph.set_axis_labels("Survival time (days)", "Number of patients")

## **Preprocessing of Test data**

### **Clinical Data**

In [ ]:
clinical_test=pd.read_csv("/content/drive/My Drive/x_test/features/clinical_data_test.csv", index_col='PatientID')
clinical_test.head()

In [ ]:
clinical_test.describe()

Variable "age" has 4 missing values

Same preprocessing than for train data is applied to test data

In [ ]:
clinical_test.Mstage.value_counts(sort=False, dropna=False)

In [ ]:
clinical_test.Histology.value_counts(sort=False, dropna=False)

In [ ]:
#Variable "Histology_cat" is created and variables "Histology" and "Mstage" are removed
clinical_test['Histology_cat']=clinical_test.Histology.apply(lambda x: 0 if x in ('Adenocarcinoma','adenocarcinoma') 
                                                              else 1 if x=='large cell' 
                                                              else 2 if x in('squamous cell carcinoma', 'Squamous cell carcinoma') 
                                                              else 3)
clinical_test=clinical_test.drop(columns=['Histology','Mstage'])

In [ ]:
clinical_test.Nstage.value_counts(sort=False, dropna=False)

There are 2 patients with Nstage="4". We don't have any in train data. These patients are grouped with Patients Nstage="3".

In [ ]:
clinical_test['Nstage']=clinical_test.Nstage.apply(lambda x: 3 if x==4 else x)

In [ ]:
clinical_test.Tstage.value_counts(sort=False, dropna=False)

In [ ]:
#Patient with Tstage="5" is grouped with patients Tstage="4"
clinical_test['Tstage']=clinical_test.Tstage.apply(lambda x: 4 if x==5 else x)

Dummification of categorical variables

In [ ]:
categories = ['Nstage', 'Tstage', 'Histology_cat']
clinical_test = pd.get_dummies(clinical_test, columns=categories, drop_first=True)
clinical_test.head()

### **Radiomics Features**

In [ ]:
radiomics_test=pd.read_csv("/content/drive/My Drive/x_test/features/radiomics_test.csv", header=1, index_col='PatientID')
radiomics_test.describe()

All the variables are complete (no missing data)

**Merge of clinical data and radiomics features**

In [ ]:
x_test_svm=pd.merge(radiomics_test, clinical_test, how='inner', left_index=True, right_index=True, sort=True)

Transformation Pipelines

In [ ]:
# Dummy variables are not going to be transformed and are thus removed temporarily from the dataset.
x_test=x_test_svm.drop(columns=['Nstage_1','Nstage_2','Nstage_3','Tstage_2','Tstage_3','Tstage_4','Histology_cat_1', 'Histology_cat_2','Histology_cat_3'])

#Same transformation as for train data is applied
x_test_ = preprocessor.transform(x_test)

x_test=pd.DataFrame(x_test_, index=x_test.index, columns=x_test.columns)
#Dummy variables are included in a new dataset in order to be merged with the other variables afterward
test=clinical_test.loc[:,['Nstage_1','Nstage_2','Nstage_3','Tstage_2','Tstage_3','Tstage_4','Histology_cat_1', 'Histology_cat_2','Histology_cat_3']]

#Merge of the transformed variables with the dummy variables
x_test=pd.merge(x_test, test, how='inner', left_index=True, right_index=True, sort=True)
x_test=x_test.rename(columns={'SourceDataset':'age','age':'SourceDataset'})
x_test.head()